In [2]:
import pandas as pd
import numpy as np
import unittest

In [ ]:
def calculate_entropy(transactions:pd.Series):
    """
    エントロピーを返す
    例）
    入力：pd.Series([Yes, Yes, No, No])
    出力：1
    """
    _, counts = np.unique(transactions, return_counts=True)
    probabilities = counts / np.sum(counts)
    entropy = - np.sum(probabilities * np.log2(probabilities))
    return entropy
    
def calculate_information_gain(transactions, attribute, class_attribute):
    """
    属性に対する情報利得を返す
    """
    total_entropy = calculate_entropy(transactions[class_attribute])
    attribute_vals, counts = np.unique(transactions[attribute], return_counts=True)

    attribute_entropy = 0
    for val, count in zip(attribute_vals, counts):
        val_subset = transactions[transactions[attribute] == val]
        attribute_entropy += (count / np.sum(counts)) * calculate_entropy(val_subset[class_attribute])

    return total_entropy - attribute_entropy

def build_id3(transactions, attributes, class_attribute):
    """決定木ID3を構成する"""

    # attributesにクラスがないならば、最も多い目的クラスを返す
    if len(attributes) == 0:
        counts = transactions[class_attribute].value_counts()
        return counts.idxmax()

    # 目的クラスが同じならば、そのクラスを返す
    unique_classes = np.unique(transactions[class_attribute])
    if len(unique_classes) == 1:
        return unique_classes[0]

    # 情報利得が最大の属性を探す
    gains = [calculate_information_gain(transactions, attribute, class_attribute) for attribute in attributes]
    max_gain_index = np.argmax(gains)
    node_attribute = attributes[max_gain_index]
    
    # 木の構成
    remaining_attributes = [attr for attr in attributes if attr != node_attribute]
    tree = {node_attribute: {}}

    for attribute_value in np.unique(transactions[node_attribute]):
        remaining_transactions = transactions[transactions[node_attribute]==attribute_value]
        tree[node_attribute][attribute_value] = build_id3(remaining_transactions, remaining_attributes, class_attribute)

    return tree         

In [ ]:
class TestID3Algorithm(unittest.TestCase):
    
    def test_calculate_entropy(self):
        # 完全に半々のデータ（エントロピーは1.0になるはず）
        data_mixed = pd.Series(['Yes', 'Yes', 'No', 'No'])
        self.assertAlmostEqual(calculate_entropy(data_mixed), 1.0)
        
        # すべて同じクラスのデータ（エントロピーは0.0になるはず）
        data_pure = pd.Series(['Yes', 'Yes', 'Yes'])
        self.assertAlmostEqual(calculate_entropy(data_pure), 0.0)


    def test_calculate_information_gain(self):
        # 特徴量Aで完璧に分割できるモックデータ
        df = pd.DataFrame({
            'Feature_A': ['X', 'X', 'Y', 'Y'],
            'Target':    ['Yes', 'Yes', 'No', 'No']
        })
        # 完璧に分割できるため、情報利得は元データの乱雑さ（1.0）と等しくなるはず
        gain = calculate_information_gain(df, 'Feature_A', 'Target')
        self.assertAlmostEqual(gain, 1.0)

    def test_id3_tree_building(self):
        # 決定木の構築テスト用のシンプルなデータセット
        df = pd.DataFrame({
            'Weather': ['Sunny', 'Sunny', 'Rainy', 'Rainy'],
            'Wind':    ['Weak', 'Strong', 'Weak', 'Strong'],
            'Play':    ['Yes', 'No', 'No', 'No'] # SunnyかつWeakの時だけYes
        })
        
        features = ['Weather', 'Wind']
        class_attribute = 'Play'
        tree = build_id3(df, features, class_attribute)
        
        # 期待される出力:
        # WeatherがRainyならすべてNo、SunnyならWindで再分割してWeak=Yes, Strong=No
        # 情報利得の計算上、最初にWeatherかWindが選ばれ、正しく辞書が構築されるか確認
        self.assertIsInstance(tree, dict)
        
    #     # 予測結果が正しいかツリーを直接辿って確認
    #     # (手動で辿れるほどシンプルなテストケースにするのがTDDのコツです)
    #     # 今回のデータだと情報利得はWeather=0.311, Wind=0.311で同点になるため、
    #     # リストの先頭にあるWeatherが選択される前提でテストします。
        self.assertEqual(tree['Weather']['Rainy'], 'No')
        self.assertEqual(tree['Weather']['Sunny']['Wind']['Weak'], 'Yes')
        self.assertEqual(tree['Weather']['Sunny']['Wind']['Strong'], 'No')

if __name__ == '__main__':
    # テストを実行
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

...
----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


In [ ]:
if __name__ == "__main__":
    # サンプルデータセット（テニスをするかどうかの判定）
    dataset = {
        'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
        'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
        'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
        'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
        'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
    }
    
    df = pd.DataFrame(dataset, columns=['Outlook', 'Temperature', 'Humidity', 'Wind', 'PlayTennis'])
    
    # 決定木の構築
    # ターゲット変数（予測したいクラス）以外のカラムを特徴量リストとして渡す
    features = df.columns[:-1].tolist()
    tree = build_id3(df, features, 'PlayTennis')
    
    # 結果の出力（辞書型で表現された決定木）
    import pprint
    print("構築された決定木:")
    pprint.pprint(tree)

構築された決定木:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}},
             'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}
